In [1]:
#Install Required Libraries
!pip install gradio transformers torchaudio librosa git+https://github.com/openai/whisper.git --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.0 MB/s eta 0:00:00


In [2]:
#Import Libraries
import gradio as gr
import whisper
from transformers import pipeline
import google.generativeai as genai
from google.colab import userdata

In [3]:
#Load whisper Model
whisper_model = whisper.load_model("base")


100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 43.1MiB/s]


In [4]:
#Load summarizer Model
bart_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [5]:
#Load google summarizer Model
action_model = pipeline("text2text-generation", model="google/flan-t5-large")
# or mistralai/Mistral-7B-Instruct
#action_model = pipeline("text2text-generation", model="mistralai/Mistral-7B-Instruct-v0.2")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [1]:
#Create Gemini model object
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

NameError: name 'genai' is not defined

In [7]:
#Define Processing Function
def process_audio(audio_file, summary_type):
    # Transcribe
    transcription = whisper_model.transcribe(audio_file)
    transcript = transcription['text']

    # Summarize
    if summary_type == "Generic Summary":
        chunks = [transcript[i:i+1000] for i in range(0, len(transcript), 1000)]
        summary = ""
        for chunk in chunks:
            result = bart_summarizer(chunk, max_length=150, min_length=40, do_sample=False)
            summary += result[0]['summary_text'] + "\n"
    elif summary_type == "Actionable Minutes":
        prompt = f"""
        Summarize the following meeting transcript. Include:
        - Key decisions
        - Action items with responsible persons
        - Deadlines if mentioned

        Transcript:
        {transcript}
        """
        result = action_model(prompt, max_length=512, do_sample=False)
        summary = result[0]['generated_text']
    elif summary_type == "Gemini Flash Summary":
        prompt = f"""
        Summarize this meeting transcript with clear action items, decisions, and follow-ups:
        {transcript}
        """
        response = gemini_model.generate_content(prompt)
        summary = response.text
    return transcript, summary

In [8]:
#Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🎙️ Meeting Minutes Generator with Multiple Summary Options")
    audio_input = gr.Audio(type="filepath", label="Upload English Audio")
    summary_type = gr.Radio(
        ["Generic Summary", "Actionable Minutes", "Gemini Flash Summary"],
        label="Choose Summary Type"
    )
    transcript_output = gr.Textbox(label="📄 Transcript", lines=10)
    summary_output = gr.Textbox(label="📝 Meeting Minutes", lines=10)
    submit_btn = gr.Button("Generate")

    submit_btn.click(fn=process_audio, inputs=[audio_input, summary_type], outputs=[transcript_output, summary_output])



In [9]:
#launch UI
demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f0da78712efe0c299.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
